In [32]:
import random
import math

random.seed(40)

def rand():
    return random.uniform(-1, 1)


def sigmoid(x):
    return 1 / (1 + math.exp(-x))


class XORNN:

    def __init__(self):
        self.w11, self.w12 = rand(), rand()
        self.w21, self.w22 = rand(), rand()
        self.b1, self.b2, self.b3 = rand(), rand(), rand()
        self.v1, self.v2 = rand(), rand()
        self.init_gradients()

    def init_gradients(self):
        self.grad_w11 = self.grad_w12 = self.grad_w21 = self.grad_w22 = 0
        self.grad_v1 = self.grad_v2 = 0
        self.grad_b1 = self.grad_b2 = self.grad_b3 = 0

    def forward_pass(self, x: tuple[int, int]) -> float:
        x1, x2 = x
        self.z1 = self.w11 * x1 + self.w12 * x2 + self.b1
        self.a1 = sigmoid(self.z1)

        self.z2 = self.w21 * x1 + self.w22 * x2 + self.b2
        self.a2 = sigmoid(self.z2)

        self.z3 = self.v1 * self.a1 + self.v2 * self.a2 + self.b3
        self.a3 = sigmoid(self.z3)

        return self.a3

    def loss_fn(self, y_hat: float, y_true: float) -> float:
        return (y_hat - y_true) ** 2

    def gradient_descent(self, epoch: int) -> None:
        if len(inputs) != len(outputs):
            raise ValueError("Inputs and outputs length don't match")
        mean_loss = 0
        for x, y in zip(inputs, outputs):
            mean_loss += self.loss_fn(self.forward_pass(x), y)
            self.accumulate_grad(x, y)
        self.mean_accumulated_grad(len(inputs))
        mean_loss /= len(inputs)
        self.update_weights()
        if epoch % 100 == 0:
            print(f"Mean loss: {mean_loss}")
            print(
                "grads:",
                round(self.grad_w11, 5),
                round(self.grad_w12, 5),
                round(self.grad_w21, 5),
                round(self.grad_w22, 5),
                round(self.grad_v1, 5),
                round(self.grad_v2, 5),
            )
            print("a1 a2 a3:", round(self.a1,3), round(self.a2,3), round(self.a3,3))
        self.init_gradients()
            

    def update_weights(self, lr=10) -> None:
        self.v1 -= lr * self.grad_v1
        self.v2 -= lr * self.grad_v2
        self.b3 -= lr * self.grad_b3
        
        self.w11 -= lr * self.grad_w11
        self.w12 -= lr * self.grad_w12
        self.b1 -= lr * self.grad_b1
        
        self.w21 -= lr * self.grad_w21
        self.w22 -= lr * self.grad_w22
        self.b2 -= lr * self.grad_b2

    def mean_accumulated_grad(self, size: int) -> None:
        self.grad_v1 /= size
        self.grad_v2 /= size
        self.grad_b3 /= size
        self.grad_w11 /= size
        self.grad_w12 /= size
        self.grad_b1 /= size
        self.grad_w21 /= size
        self.grad_w22 /= size
        self.grad_b2 /= size

    def accumulate_grad(self, x: tuple[int, int], y_true: float) -> None:
        x1, x2 = x
        dL_da3 = 2 * (self.a3 - y_true)
        da3_dz3 = self.a3 * (1 - self.a3)
        # delta at the last (= output) neuron
        delta1 = dL_da3 * da3_dz3

        # final multipliers and update last layer's gradients for the only neuron
        dz3_dv1 = self.a1
        dz3_dv2 = self.a2
        dz3_db3 = 1
        self.grad_v1 += delta1 * dz3_dv1
        self.grad_v2 += delta1 * dz3_dv2
        self.grad_b3 += delta1 * dz3_db3

        # delta2 is at the 1st neuron of middle (= hidden) layer
        dz3_da1 = self.v1
        da1_dz1 = self.a1 * (1 - self.a1)
        delta2 = delta1 * dz3_da1 * da1_dz1

        # delta3 is at the 2nd neuron of middle layer
        dz3_da2 = self.v2
        da2_dz2 = self.a2 * (1 - self.a2)
        delta3 = delta1 * dz3_da2 * da2_dz2

        # final differentials at 1st neuron of middle layer and accumulate
        dz1_dw11 = x1
        dz1_dw12 = x2
        dz1_db1 = 1
        self.grad_w11 += delta2 * dz1_dw11
        self.grad_w12 += delta2 * dz1_dw12
        self.grad_b1 += delta2 * dz1_db1

        # final differentials at 2nd neuron of middle layer and accumulate
        self.grad_w21 += delta3 * x1
        self.grad_w22 += delta3 * x2
        self.grad_b2 += delta3
        

    
inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]
outputs = [0, 1, 1, 0]
xor = XORNN()

"""
Performing training here.
Experimented with lr (Learning Rate) of 1. Still had to do about 2000 epochs
 to achieve desired testing result.
However, increasing learning rate to 10, even 300 epochs are sufficient.
Even with such high learning rate, there aren't any oscillations, 
 since gradients themselves are very small,
 e.g.: grad_w11 = 0.00332, grad_v2 = -0.02711 after 100 epochs.
"""
for epoch in range(300):
    xor.gradient_descent(epoch)

for x in inputs:
    print(xor.forward_pass(x))

Mean loss: 0.2767243531836844
grads: 0.00332 0.00156 -0.00798 -0.00915 -0.05523 -0.02711
a1 a2 a3: 0.832 0.261 0.311
Mean loss: 0.25371232520925624
grads: 0.003 0.0001 -0.00124 -0.00159 -0.01973 -0.01599
a1 a2 a3: 0.652 0.387 0.422
Mean loss: 0.01034023042568785
grads: 0.00127 -0.00095 0.00136 -0.00164 0.00289 -0.00351
a1 a2 a3: 0.949 0.072 0.088
0.04942748567374576
0.9528092172351027
0.9460128816988678
0.0429704295169362
